# Phase 3 — Churn Prediction Modeling

Logistic Regression → XGBoost → LightGBM → SHAP


In [2]:
!pip install xgboost lightgbm shap

  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached slicer-0.0.8-py3-none-any.whl.metadata (4.0 kB)
  Using cached numba-0.65.1-cp314-cp314-win_amd64.whl.metadata (3.0 kB)
  Using cached llvmlite-0.47.0-cp314-cp314-win_amd64.whl.metadata (5.1 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
    --------------------------------------- 1.3/101.7 MB 4.5 MB/s eta 0:00:23
    --------------------------------------- 2.4/101.7 MB 4.5 MB/s eta 0:00:23
   - -------------------------------------- 3.1/101.7 MB 4.5 MB/s eta 0:00:22
   - -------------------------------------- 3.9/101.7 MB 4.4 MB/s eta 0:00:23
   - -------------------------------------- 4.7/101.7 MB 4.1 MB/s eta 0:00:24
   -- ------------------------------------- 5.8/101.7 MB 4.3 MB/s eta 0:00:23
   -- ------------------------------------- 6.8/101


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, f1_score,
                             precision_recall_curve, average_precision_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import shap

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f8f8',
    'axes.grid': True, 'grid.color': 'white', 'grid.linewidth': 1.2,
    'font.size': 11, 'axes.titlesize': 13, 'axes.titleweight': 'bold',
})
BLUE, RED, GREEN, PURPLE = '#4C72B0', '#DD3524', '#2CA02C', '#9467BD'

print("=" * 55)
print("  PHASE 3 — CHURN PREDICTION MODELING")
print("=" * 55)

# ── STEP 1: Feature engineering ───────────────────────────────
print("\n[1/6] Loading & engineering features...")
df = pd.read_csv('data/Customer-Churn-Records.csv')
df.columns = [c.strip().replace(' ', '_') for c in df.columns]
df.drop(columns=['RowNumber','CustomerId','Surname','Complain'], inplace=True)

df['age_band']                = pd.cut(df['Age'],bins=[17,27,37,47,57,67,100],
                                    labels=['18-28','28-38','38-48','48-58','58-68','68+'])
df['is_peak_churn_age']       = df['Age'].between(48,57).astype(int)
df['has_zero_balance']        = (df['Balance']==0).astype(int)
df['log_balance']             = np.log1p(df['Balance'])
df['log_salary']              = np.log1p(df['EstimatedSalary'])
df['balance_to_salary_ratio'] = np.where(df['EstimatedSalary']==0, 0,
                                          df['Balance']/df['EstimatedSalary'])
df['is_high_risk_products']   = (df['NumOfProducts']>=3).astype(int)
df['is_single_product']       = (df['NumOfProducts']==1).astype(int)
df['is_new_customer']         = (df['Tenure']==0).astype(int)
# tenure=0 means new customer; fill products_per_year with NumOfProducts
df['products_per_year']       = np.where(df['Tenure']==0,
                                          df['NumOfProducts'].astype(float),
                                          df['NumOfProducts']/df['Tenure'])
df['engagement_score']        = (df['IsActiveMember'] + df['HasCrCard'] +
                                  (df['Satisfaction_Score']>=4).astype(int))
df['is_disengaged']           = ((df['IsActiveMember']==0)&
                                  (df['Satisfaction_Score']<=2)).astype(int)
df['geography_risk_score']    = df['Geography'].map({'Germany':3,'Spain':2,'France':1})
df['is_high_churn_geo']       = (df['Geography']=='Germany').astype(int)
df['is_poor_credit']          = (df['CreditScore']<580).astype(int)
df['card_tier_rank']          = df['Card_Type'].str.upper().map(
                                  {'GOLD':1,'SILVER':2,'PLATINUM':3,'DIAMOND':4})
df['is_german_female']        = ((df['Geography']=='Germany')&
                                  (df['Gender']=='Female')).astype(int)
df['is_inactive_high_prod']   = ((df['IsActiveMember']==0)&
                                  (df['NumOfProducts']>=3)).astype(int)

le = LabelEncoder()
df['Gender_enc']    = le.fit_transform(df['Gender'])
df['Geography_enc'] = le.fit_transform(df['Geography'])
df['age_band_enc']  = le.fit_transform(df['age_band'].astype(str))
df['card_type_enc'] = le.fit_transform(df['Card_Type'].str.upper())

FEATURE_COLS = [
    'CreditScore','Age','Tenure','Balance','NumOfProducts',
    'HasCrCard','IsActiveMember','EstimatedSalary','Satisfaction_Score','Point_Earned',
    'is_peak_churn_age','has_zero_balance','log_balance','log_salary',
    'balance_to_salary_ratio','is_high_risk_products','is_single_product',
    'is_new_customer','products_per_year','engagement_score','is_disengaged',
    'geography_risk_score','is_high_churn_geo','is_poor_credit',
    'card_tier_rank','is_german_female','is_inactive_high_prod',
    'Gender_enc','Geography_enc','card_type_enc',
]
X = df[FEATURE_COLS]
y = df['Exited']
assert X.isna().sum().sum() == 0, f"NaN found: {X.isna().sum()[X.isna().sum()>0]}"
print(f"   Shape: {X.shape} | NaN: 0 confirmed")

# ── STEP 2: Split ─────────────────────────────────────────────
print("\n[2/6] Splitting data...")
X_train,X_test,y_train,y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f"   Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")
print(f"   Churn rate — Train: {y_train.mean():.1%} | Test: {y_test.mean():.1%}")

# ── STEP 3: Train models ──────────────────────────────────────
print("\n[3/6] Training models...")

def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:,1]
    auc    = roc_auc_score(y_te, y_prob)
    f1     = f1_score(y_te, y_pred)
    cv_auc = cross_val_score(model, X_tr, y_tr,
                              cv=StratifiedKFold(5,shuffle=True,random_state=42),
                              scoring='roc_auc').mean()
    print(f"   {name:<22} AUC={auc:.4f}  F1={f1:.4f}  CV-AUC={cv_auc:.4f}")
    return {'name':name,'model':model,'y_pred':y_pred,'y_prob':y_prob,
            'auc':auc,'f1':f1,'cv_auc':cv_auc}

results = {}

lr = Pipeline([('scaler',StandardScaler()),
               ('clf',LogisticRegression(class_weight='balanced',
                                          max_iter=1000,random_state=42))])
lr.fit(X_train, y_train)
results['Logistic Regression'] = evaluate('Logistic Regression',lr,X_train,y_train,X_test,y_test)

scale_pos = (y_train==0).sum()/(y_train==1).sum()
xgb = XGBClassifier(n_estimators=400,max_depth=5,learning_rate=0.05,
                     subsample=0.8,colsample_bytree=0.8,
                     scale_pos_weight=scale_pos,
                     eval_metric='auc',random_state=42,verbosity=0)
xgb.fit(X_train, y_train)
results['XGBoost'] = evaluate('XGBoost',xgb,X_train,y_train,X_test,y_test)

lgbm = LGBMClassifier(n_estimators=400,max_depth=5,learning_rate=0.05,
                       subsample=0.8,colsample_bytree=0.8,
                       class_weight='balanced',random_state=42,verbosity=-1)
lgbm.fit(X_train, y_train)
results['LightGBM'] = evaluate('LightGBM',lgbm,X_train,y_train,X_test,y_test)

best_name = max(results, key=lambda k: results[k]['auc'])
best      = results[best_name]
print(f"\n   Best model: {best_name} (AUC={best['auc']:.4f})")

# ── STEP 4: Plots ─────────────────────────────────────────────
print("\n[4/6] Generating evaluation plots...")
colors = {'Logistic Regression':PURPLE,'XGBoost':RED,'LightGBM':GREEN}

fig, axes = plt.subplots(1,3,figsize=(18,5))
fig.suptitle('Model Evaluation',fontsize=15,fontweight='bold')

ax = axes[0]
for name,r in results.items():
    fpr,tpr,_ = roc_curve(y_test, r['y_prob'])
    ax.plot(fpr,tpr,color=colors[name],lw=2,
            label=f"{name} ({r['auc']:.3f})")
ax.plot([0,1],[0,1],'k--',lw=1)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves'); ax.legend(fontsize=9)

ax = axes[1]
for name,r in results.items():
    prec,rec,_ = precision_recall_curve(y_test,r['y_prob'])
    ap = average_precision_score(y_test,r['y_prob'])
    ax.plot(rec,prec,color=colors[name],lw=2,label=f"{name} (AP={ap:.3f})")
ax.axhline(y_test.mean(),color='k',linestyle='--',lw=1,
           label=f'Baseline ({y_test.mean():.2f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall'); ax.legend(fontsize=9)

ax = axes[2]
names = list(results.keys())
aucs  = [results[n]['auc'] for n in names]
f1s   = [results[n]['f1']  for n in names]
x     = np.arange(len(names)); w=0.35
bars1 = ax.bar(x-w/2,aucs,w,label='ROC-AUC',color=[colors[n] for n in names],alpha=0.85)
bars2 = ax.bar(x+w/2,f1s, w,label='F1',     color=[colors[n] for n in names],alpha=0.5)
ax.set_xticks(x); ax.set_xticklabels([n.replace(' ','\n') for n in names],fontsize=9)
ax.set_ylim(0.5,1.0); ax.set_title('AUC vs F1'); ax.legend()
for b in list(bars1)+list(bars2):
    ax.text(b.get_x()+b.get_width()/2,b.get_height()+0.005,
            f'{b.get_height():.3f}',ha='center',va='bottom',fontsize=8)
plt.tight_layout()
plt.savefig('data/plot_p3_01_model_comparison.png',dpi=150,bbox_inches='tight')
plt.close()
print("   Saved: plot_p3_01_model_comparison.png")

fig,axes = plt.subplots(1,3,figsize=(15,4))
fig.suptitle('Confusion Matrices (Test Set)',fontsize=14,fontweight='bold')
for ax,(name,r) in zip(axes,results.items()):
    cm = confusion_matrix(y_test,r['y_pred'])
    sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',ax=ax,
                xticklabels=['Stayed','Churned'],yticklabels=['Stayed','Churned'])
    ax.set_title(f"{name}\nAUC={r['auc']:.3f}  F1={r['f1']:.3f}")
    ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('data/plot_p3_02_confusion_matrices.png',dpi=150,bbox_inches='tight')
plt.close()
print("   Saved: plot_p3_02_confusion_matrices.png")

# ── STEP 5: Feature importance + SHAP ────────────────────────
print("\n[5/6] Feature importance & SHAP...")
fig,axes = plt.subplots(1,2,figsize=(18,7))
fig.suptitle('Feature Importance',fontsize=14,fontweight='bold')
for ax,(name,model) in zip(axes,[('XGBoost',xgb),('LightGBM',lgbm)]):
    imp = pd.Series(model.feature_importances_,index=FEATURE_COLS).sort_values().tail(20)
    imp.plot(kind='barh',ax=ax,color=colors[name],alpha=0.8)
    ax.set_title(f'{name} — top 20 features')
    ax.set_xlabel('Importance score')
plt.tight_layout()
plt.savefig('data/plot_p3_03_feature_importance.png',dpi=150,bbox_inches='tight')
plt.close()
print("   Saved: plot_p3_03_feature_importance.png")

print("   Computing SHAP values (~30 sec)...")
raw_model = best['model']
if hasattr(raw_model,'named_steps'):
    raw_model = raw_model.named_steps['clf']
explainer = shap.TreeExplainer(raw_model)
shap_vals  = explainer.shap_values(X_test)
if isinstance(shap_vals,list):
    shap_vals = shap_vals[1]

fig,axes = plt.subplots(1,2,figsize=(18,7))
fig.suptitle(f'SHAP Explainability — {best_name}',fontsize=14,fontweight='bold')
plt.sca(axes[0])
shap.summary_plot(shap_vals,X_test,plot_type='bar',max_display=15,show=False)
axes[0].set_title('Mean |SHAP| — global importance')
plt.sca(axes[1])
shap.summary_plot(shap_vals,X_test,max_display=15,show=False)
axes[1].set_title('SHAP value distribution')
plt.tight_layout()
plt.savefig('data/plot_p3_04_shap.png',dpi=150,bbox_inches='tight')
plt.close()
print("   Saved: plot_p3_04_shap.png")

# ── STEP 6: Summary ───────────────────────────────────────────
print("\n[6/6] Summary")
print("-"*55)
for name,r in results.items():
    print(f"  {name:<22} AUC={r['auc']:.4f}  F1={r['f1']:.4f}  CV={r['cv_auc']:.4f}")
print("-"*55)
print(f"\n  Best : {best_name}")
print(f"  AUC  : {best['auc']:.4f}")
print(f"  F1   : {best['f1']:.4f}")
print("\n  Phase 3 complete.")


  PHASE 3 — CHURN PREDICTION MODELING

[1/6] Loading & engineering features...
   Shape: (10000, 30) | NaN: 0 confirmed

[2/6] Splitting data...
   Train: 8000 | Test: 2000
   Churn rate — Train: 20.4% | Test: 20.4%

[3/6] Training models...
   Logistic Regression    AUC=0.8500  F1=0.5776  CV-AUC=0.8342
   XGBoost                AUC=0.8687  F1=0.6266  CV-AUC=0.8522
   LightGBM               AUC=0.8669  F1=0.6168  CV-AUC=0.8513

   Best model: XGBoost (AUC=0.8687)

[4/6] Generating evaluation plots...
   Saved: plot_p3_01_model_comparison.png
   Saved: plot_p3_02_confusion_matrices.png

[5/6] Feature importance & SHAP...
   Saved: plot_p3_03_feature_importance.png
   Computing SHAP values (~30 sec)...
   Saved: plot_p3_04_shap.png

[6/6] Summary
-------------------------------------------------------
  Logistic Regression    AUC=0.8500  F1=0.5776  CV=0.8342
  XGBoost                AUC=0.8687  F1=0.6266  CV=0.8522
  LightGBM               AUC=0.8669  F1=0.6168  CV=0.8513
---------------